In [2]:
import logging
logger = logging.getLogger('GPUCheck')
logger.warning("No GPUs found; training will run on CPU.")


import tensorflow as tf

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)  # optional: avoid preallocating all memory
    logger.info(f"Using GPUs: {gpus}")
else:
    logger.warning("No GPUs found; training will run on CPU.")


No GPUs found; training will run on CPU.
No GPUs found; training will run on CPU.


In [8]:
import random
import shutil
from pathlib import Path

# ------------------------------------------------------------------
# Paths and parameters
# ------------------------------------------------------------------
images_dir = Path('/home/rbielski/Atlas_2/Training/Images')
masks_dir  = Path('/home/rbielski/Atlas_2/Training/Masks')
output_root = Path('/home/rbielski/Atlas_2/Training_Split')  # sibling to Training/

train_frac = 0.8  # 80% of pairs for training
seed = 42

# Tokens indicating modality or mask descriptors
image_tokens = ['_T1w', '_T1', '_t1', '_T2w', '_t2',
                '_flair', '_FLAIR', '_dwi', '_DWI',
                '_adc', '_ADC', '_image', '_brain']
mask_tokens  = ['_mask', '_lesion', '_label', '_seg', '_desc']

def strip_at_first_token(name, tokens):
    indices = [name.find(tok) for tok in tokens if tok in name]
    return name[:min(indices)] if indices else name

def find_pairs(images_dir, masks_dir):
    """Identify image–mask pairs by matching shared prefixes."""
    image_files = list(images_dir.rglob('*.nii.gz'))
    mask_files  = list(masks_dir.rglob('*.nii.gz'))
    pairs = []
    for mask_path in mask_files:
        mask_base = strip_at_first_token(mask_path.stem, mask_tokens)
        match = None
        for img_path in image_files:
            img_base = strip_at_first_token(img_path.stem, image_tokens)
            if img_base == mask_base or img_base in mask_base or mask_base in img_base:
                match = img_path
                break
        if match:
            pairs.append((match, mask_path))
    return pairs

def split_pairs(pairs, train_frac=0.8, seed=42):
    random.seed(seed)
    pairs_shuffled = pairs.copy()
    random.shuffle(pairs_shuffled)
    n_train = int(len(pairs_shuffled) * train_frac)
    return pairs_shuffled[:n_train], pairs_shuffled[n_train:]

def copy_pairs(pairs, dest_images: Path, dest_masks: Path):
    dest_images.mkdir(parents=True, exist_ok=True)
    dest_masks.mkdir(parents=True, exist_ok=True)
    for img_path, mask_path in pairs:
        shutil.copy2(img_path, dest_images / img_path.name)
        shutil.copy2(mask_path, dest_masks / mask_path.name)

# ------------------------------------------------------------------
# Execute the splitting
# ------------------------------------------------------------------
if not images_dir.exists() or not masks_dir.exists():
    raise FileNotFoundError("Could not find the specified Images or Masks directories.")

pairs = find_pairs(images_dir, masks_dir)
if not pairs:
    raise RuntimeError("No image–mask pairs could be identified. Check your filenames.")

train_pairs, test_pairs = split_pairs(pairs, train_frac=train_frac, seed=seed)

# Create split structure under output_root
train_img_dir = output_root / 'Training_Set' / 'Images'
train_msk_dir = output_root / 'Training_Set' / 'Masks'
test_img_dir  = output_root / 'Test_set'    / 'Images'
test_msk_dir  = output_root / 'Test_set'    / 'Masks'

print(f'Copying {len(train_pairs)} pairs into {train_img_dir.parent}…')
copy_pairs(train_pairs, train_img_dir, train_msk_dir)

print(f'Copying {len(test_pairs)} pairs into {test_img_dir.parent}…')
copy_pairs(test_pairs, test_img_dir, test_msk_dir)

print('✅ Structured dataset split complete.')


Copying 524 pairs into /home/rbielski/Atlas_2/Training_Split/Training_Set…
Copying 131 pairs into /home/rbielski/Atlas_2/Training_Split/Test_set…
✅ Structured dataset split complete.


In [ ]:

"""
SMART SOTA 2025: Stroke Lesion Segmentation (Dynamic Input Version)

This training script builds upon prior production and cropped variants but adds
support for arbitrary volumetric input sizes.  It automatically determines
the largest spatial dimensions present in a dataset and pads smaller volumes
so that all inputs share the same shape.  The script retains detailed
logging, memory monitoring, data augmentation and custom layers from the
previous versions while incorporating recommendations from the latest model
evaluation:

* Dice/boundary loss weights adjusted to emphasise boundary precision
* Over‑segmentation mitigation via adjustable decision threshold
* Slightly stronger augmentation (rotations/flips/gamma) when overfitting
  is suspected
* Tunable L2 regularisation and dropout rates
* Longer warm‑up and lower minimum learning rate

The batch size is fixed at 2, but input volumes may be of any shape as long
as the corresponding mask has identical dimensions.  All three spatial
dimensions are padded to the maximum observed size for consistent training.
"""

import os
import sys
import logging
from pathlib import Path

# ---------------------------------------------------------------------------
# Environment configuration
# ---------------------------------------------------------------------------
# Ensure logs and callbacks directories exist before TensorFlow imports
os.makedirs("logs", exist_ok=True)
os.makedirs("callbacks", exist_ok=True)

# Configure TensorFlow behaviour and GPU settings.  Many of these
# environment variables mirror those used in earlier scripts to minimise
# CUDNN/XLA related errors on heterogeneous GPU clusters.
os.environ['TF_GPU_ALLOCATOR'] = 'cuda_malloc_async'
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'      # Suppress excessive warnings
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'
os.environ['TF_XLA_FLAGS'] = '--tf_xla_enable_xla_devices=false'
os.environ['XLA_FLAGS'] = '--xla_gpu_cuda_data_dir=/usr/local/cuda'

# ---------------------------------------------------------------------------
# Logging configuration
# ---------------------------------------------------------------------------
# Use two file handlers and one stream handler.  One file captures all
# high‑level events (INFO and above) and the other captures per‑process
# debugging output.  A console stream is kept for quick feedback.
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler('logs/smart_sota_dynamic.log'),
        logging.FileHandler(f'logs/training_dynamic_{os.getpid()}.debug.log'),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger('SmartSOTA_Dynamic')
logger.setLevel(logging.DEBUG)

# ---------------------------------------------------------------------------
# Imports with graceful degradation
# ---------------------------------------------------------------------------
try:
    import tensorflow as tf
    from tensorflow.keras import layers
    import numpy as np
    import nibabel as nib
    from scipy.ndimage import zoom, binary_dilation, rotate
    from sklearn.model_selection import KFold
    from skimage import measure
    import json
    import time
    import math
    import random
    import gc
    import psutil
    from functools import lru_cache
    from concurrent.futures import ThreadPoolExecutor
    logger.info("✅ All imports successful")
except ImportError as e:
    logger.critical(f"❌ Import failed: {e}")
    sys.exit(1)

# ---------------------------------------------------------------------------
# Runtime configuration
# ---------------------------------------------------------------------------
# Disable eager execution for performance and to avoid certain CUDNN issues.
tf.config.run_functions_eagerly(False)
logger.info(f"TensorFlow eager execution: {tf.executing_eagerly()}")

warnings_to_ignore = [UserWarning, DeprecationWarning, FutureWarning]
for w in warnings_to_ignore:
    tf.autograph.set_verbosity(0)
import warnings
for w in warnings_to_ignore:
    warnings.filterwarnings("ignore", category=w)
warnings.filterwarnings("ignore", module="nibabel")

logger.info(
    f"Environment verified:\n"
    f"- Python {sys.version}\n"
    f"- TensorFlow {tf.__version__}\n"
    f"- NumPy {np.__version__}\n"
    f"- GPU devices: {len(tf.config.list_physical_devices('GPU'))}"
)

# ---------------------------------------------------------------------------
# Training configuration
# ---------------------------------------------------------------------------
class DynamicTrainingConfig:
    """Configuration class supporting variable input shapes.

    The first call to `detect_input_shape` will populate INPUT_SHAPE based on
    observed dataset maxima.  All other hyperparameters may be tuned from
    outside to reflect recommendations from prior evaluations.
    """

    # Root directory containing Images and Masks (subdirectories or mixed)
    DATA_DIR: Path = Path("/home/rbielski/Atlas_2/Training_Split/Training_Set")

    # The input shape will be set by detect_input_shape; default to None
    INPUT_SHAPE = None

    # Data split configuration
    VALIDATION_SPLIT = 0.15
    SMALL_LESION_THRESHOLD = 100

    # Training schedule
    BATCH_SIZE = 2
    INITIAL_EPOCH = 0
    TOTAL_EPOCHS = 200
    INITIAL_LR = 1e-4
    MIN_LR = 5e-7               # Lower minimum LR for potential late training plateaus
    WARMUP_EPOCHS = 15          # Extended warmup
    MAX_GRAD_NORM = 1.0

    # Model architecture
    BASE_FILTERS = 8
    DROPOUT_RATE = 0.55         # Balanced dropout for robustness
    L2_REG = 1.5e-3             # Moderate L2 regularisation
    MAMBA_DEPTH = 2
    SAM_HEADS = 4

    # Data augmentation
    AUGMENTATION_INTENSITY = 0.5
    SYNTHETIC_LESION_PROB = 0.3
    ROTATION_RANGE = 20         # Slightly larger range than original

    # Loss configuration (based on evaluation recommendations)
    USE_BOUNDARY_LOSS = True
    DICE_LOSS_WEIGHT = 0.4
    BOUNDARY_LOSS_WEIGHT = 0.6
    DEEP_SUPERVISION_WEIGHTS = [0.1, 0.2, 0.3]

    # Decision threshold for converting logits to binary masks
    DECISION_THRESHOLD = 0.55    # Raise threshold to mitigate over‑segmentation

    # Output directories
    MODEL_DIR = Path("models/dynamic_production")
    CALLBACKS_DIR = Path("callbacks/dynamic_production")

    def __init__(self):
        # Timestamp for saving model checkpoints
        self.timestamp = time.strftime("%Y%m%d_%H%M%S")
        self.MODEL_DIR.mkdir(parents=True, exist_ok=True)
        self.CALLBACKS_DIR.mkdir(parents=True, exist_ok=True)
        # Write config to JSON for reproducibility
        config_dict = {k: v for k, v in self.__dict__.items()
                      if not k.startswith('__') and not callable(v)}
        with open(self.MODEL_DIR / "config.json", "w") as f:
            json.dump(config_dict, f, indent=2, default=str)
        logger.info(
            f"📋 DynamicTrainingConfig initialised:\n"
            f"   Data directory: {self.DATA_DIR}\n"
            f"   Batch size: {self.BATCH_SIZE}\n"
            f"   Warmup epochs: {self.WARMUP_EPOCHS}\n"
            f"   Minimum LR: {self.MIN_LR}\n"
            f"   Dice/boundary weights: {self.DICE_LOSS_WEIGHT}:{self.BOUNDARY_LOSS_WEIGHT}\n"
        )

    @property
    def model_path(self) -> Path:
        return self.MODEL_DIR / f"smart_sota_dynamic_{self.timestamp}.keras"

    @property
    def checkpoint_path(self) -> Path:
        return self.CALLBACKS_DIR / "best_model_dynamic.keras"


# ---------------------------------------------------------------------------
# Custom layers (identical to previous implementations)
# ---------------------------------------------------------------------------
class ResidualConvBlock(layers.Layer):
    """Simplified residual block to avoid CUDNN gradient issues"""
    def __init__(self, filters, kernel_reg=None, **kwargs):
        super().__init__(**kwargs)
        self.filters = filters
        self.kernel_reg = kernel_reg

    def build(self, input_shape):
        self.conv1 = layers.Conv3D(self.filters, 3, padding='same',
                                   kernel_regularizer=self.kernel_reg)
        self.bn1 = layers.BatchNormalization()
        self.conv2 = layers.Conv3D(self.filters, 3, padding='same',
                                   kernel_regularizer=self.kernel_reg)
        self.bn2 = layers.BatchNormalization()
        self.dropout = layers.SpatialDropout3D(0.1)
        # Always project if channel mismatch
        self.residual_conv = layers.Conv3D(self.filters, 1, padding='same')
        self.residual_bn = layers.BatchNormalization()
        super().build(input_shape)

    def call(self, inputs, training=None):
        x = self.conv1(inputs)
        x = self.bn1(x, training=training)
        x = tf.nn.relu(x)
        x = self.dropout(x, training=training)
        x = self.conv2(x)
        x = self.bn2(x, training=training)
        # Residual connection
        residual = self.residual_conv(inputs)
        residual = self.residual_bn(residual, training=training)
        return tf.nn.relu(x + residual)

    def get_config(self):
        config = super().get_config()
        config.update({
            "filters": self.filters,
            "kernel_reg": tf.keras.regularizers.serialize(self.kernel_reg)
                           if self.kernel_reg else None
        })
        return config


class VisionMambaBlock(layers.Layer):
    """Efficient vision Mamba block with dynamic input support"""
    def __init__(self, filters, kernel_size=3, expansion=2, **kwargs):
        super().__init__(**kwargs)
        self.filters = filters
        self.kernel_size = kernel_size
        self.expansion = expansion

    def build(self, input_shape):
        self.in_conv = layers.Conv3D(
            self.filters * self.expansion,
            1,
            use_bias=False,
            padding='same',
            data_format='channels_last')
        self.spatial_conv = layers.Conv3D(
            self.filters * self.expansion,
            self.kernel_size,
            padding='same',
            use_bias=False,
            data_format='channels_last')
        self.out_conv = layers.Conv3D(
            self.filters, 1,
            padding='same',
            data_format='channels_last')
        self.norm = layers.LayerNormalization()
        self.dropout = layers.SpatialDropout3D(0.1)
        super().build(input_shape)

    def call(self, inputs, training=None):
        x = self.in_conv(inputs)
        x = tf.nn.relu(x)
        x = self.spatial_conv(x)
        x = tf.cast(tf.nn.relu(x), inputs.dtype)
        x = self.dropout(x, training=training)
        x = self.out_conv(x)
        x = self.norm(x)
        return x + inputs

    def get_config(self):
        config = super().get_config()
        config.update({
            "filters": self.filters,
            "kernel_size": self.kernel_size,
            "expansion": self.expansion
        })
        return config


class SAM2Attention(layers.Layer):
    """Enhanced SAM2 attention with hierarchical memory banks"""
    def __init__(self, filters, heads, **kwargs):
        super().__init__(**kwargs)
        self.filters = filters
        self.heads = heads
        self.depth = filters // heads
        if filters % heads != 0:
            raise ValueError("Filters must be divisible by heads")

    def build(self, input_shape):
        self.query = layers.Conv3D(self.filters, 1, data_format='channels_last')
        self.key = layers.Conv3D(self.filters, 1, data_format='channels_last')
        self.value = layers.Conv3D(self.filters, 1, data_format='channels_last')
        self.out_conv = layers.Conv3D(input_shape[-1], 1, data_format='channels_last')
        self.memory_bank = self.add_weight(
            name='memory_bank',
            shape=(1, 1, 1, 1, self.filters),
            initializer='zeros',
            trainable=True
        )
        self.dropout = layers.SpatialDropout3D(0.1)
        super().build(input_shape)

    def call(self, inputs, training=None):
        batch_size = tf.shape(inputs)[0]
        height = tf.shape(inputs)[1]
        width = tf.shape(inputs)[2]
        depth_dim = tf.shape(inputs)[3]
        q = self.query(inputs)
        k = self.key(inputs)
        v = self.value(inputs)
        k = k + self.memory_bank
        v = v + self.memory_bank
        q = self._split_heads_safe(q, batch_size, height, width, depth_dim)
        k = self._split_heads_safe(k, batch_size, height, width, depth_dim)
        v = self._split_heads_safe(v, batch_size, height, width, depth_dim)
        dk = tf.cast(self.depth, q.dtype)
        attn_logits = tf.matmul(q, k, transpose_b=True)
        attn_logits = attn_logits / tf.math.sqrt(dk)
        attn_weights = tf.nn.softmax(attn_logits, axis=-1)
        attn_output = tf.matmul(attn_weights, v)
        attn_output = self._combine_heads_safe(attn_output, batch_size, height, width, depth_dim)
        output = self.out_conv(attn_output)
        output = self.dropout(output, training=training)
        return output + inputs

    def _split_heads_safe(self, x, batch_size, height, width, depth_dim):
        x = tf.reshape(x, [batch_size, height, width, depth_dim, self.heads, self.depth])
        return tf.transpose(x, perm=[0, 4, 1, 2, 3, 5])

    def _combine_heads_safe(self, x, batch_size, height, width, depth_dim):
        x = tf.transpose(x, perm=[0, 2, 3, 4, 1, 5])
        return tf.reshape(x, [batch_size, height, width, depth_dim, self.filters])

    def get_config(self):
        config = super().get_config()
        config.update({
            "filters": self.filters,
            "heads": self.heads
        })
        return config


# ---------------------------------------------------------------------------
# Utility functions for memory monitoring
# ---------------------------------------------------------------------------
def log_memory_usage(stage: str) -> None:
    process = psutil.Process(os.getpid())
    gb_used = process.memory_info().rss / 1024**3
    gpu_mem = []
    try:
        for i in range(4):
            alloc = tf.config.experimental.get_memory_info(f'GPU:{i}')
            gpu_mem.append(f"GPU{i}: {alloc['current']/1e9:.2f}GB")
    except Exception:
        gpu_mem = ["GPU mem tracking failed"]
    try:
        disk_usage = psutil.disk_usage('/')
        disk_free_gb = disk_usage.free / 1024**3
        disk_info = f"Disk: {disk_free_gb:.1f}GB free"
    except Exception:
        disk_info = "Disk: unavailable"
    logger.info(f"Memory at {stage}: CPU={gb_used:.2f}GB | {' | '.join(gpu_mem)} | {disk_info}")


# ---------------------------------------------------------------------------
# Dataset inspection and loading
# ---------------------------------------------------------------------------
from pathlib import Path
import numpy as np
import nibabel as nib
import gc



def detect_input_shape(data_dir: Path) -> tuple:
    """
    Determine the maximum spatial shape of images in the dataset and round each
    dimension to the nearest multiple of 16.  If no valid 3‑D NIfTI volumes are
    found, raise an error summarising the encountered issues.

    A NIfTI file is considered valid if it loads without error and has at
    least three dimensions (e.g. shape (X, Y, Z) or (X, Y, Z, C)).
    """
    logger.info("🔍 Detecting input shape from dataset…")
    image_files = list(data_dir.rglob("*.nii.gz"))[:50]
    max_shape = [0, 0, 0]
    invalid_files = []  # collect reasons for invalid files

    if not image_files:
        raise FileNotFoundError(f"No .nii.gz files found under {data_dir}")

    for f in image_files:
        try:
            img = nib.load(str(f))
            shape = img.shape
            # Require at least three spatial dimensions
            if len(shape) >= 3:
                for i in range(3):
                    max_shape[i] = max(max_shape[i], shape[i])
            else:
                invalid_files.append(f"{f.name}: shape {shape} has fewer than 3 dimensions")
        except Exception as e:
            invalid_files.append(f"{f.name}: failed to load ({e})")

    # If no valid images were found, report why
    if all(dim == 0 for dim in max_shape):
        error_msg = (
            f"No valid 3‑D NIfTI files found in {data_dir}. "
            "The following issues were encountered:\n  - "
            + "\n  - ".join(invalid_files)
        )
        raise RuntimeError(error_msg)

    # Round each dimension to nearest multiple of 16
    def nearest_multiple_of_16(x):
        return int(round(x / 16.0) * 16)

    rounded_shape = tuple(nearest_multiple_of_16(dim) for dim in max_shape)
    logger.info(
        f"📐 Detected max volume dimensions: {tuple(max_shape)} → "
        f"rounded to nearest multiple of 16: {rounded_shape}"
    )
    return rounded_shape


def load_generic_dataset(config: DynamicTrainingConfig):
    """
    Load images and masks from a dataset with arbitrary structures.

    This function finds image and mask files by pattern matching (e.g. files
    containing 'T1w', 't1', 'image' for images, and 'mask', 'lesion', 'label'
    for masks). It strips off modality or descriptor suffixes (like '_T1w',
    '_label-L_desc-', etc.) to obtain a shared subject/session prefix and then
    performs a fuzzy match to pair each mask with its corresponding image.
    It returns a list of (image_path, mask_path) tuples and a vector
    indicating lesion presence for stratified splitting.
    """
    logger.info("📚 Loading generic dataset...")
    log_memory_usage("dataset_load_start")

    data_dir = config.DATA_DIR
    if not data_dir.exists():
        raise FileNotFoundError(f"Data directory not found: {data_dir}")

    # Glob patterns for locating images and masks
    image_patterns = ["*T1w*.nii.gz", "*t1*.nii.gz", "*image*.nii.gz", "*brain*.nii.gz"]
    mask_patterns  = ["*mask*.nii.gz", "*lesion*.nii.gz", "*label*.nii.gz", "*seg*.nii.gz"]

    images = []
    masks  = []

    # Find image files: stop at the first pattern that yields results
    for pattern in image_patterns:
        images = list(data_dir.rglob(pattern))
        if images:
            logger.info(f"✅ Found {len(images)} images with pattern '{pattern}'")
            break

    # Find mask files: stop at the first pattern that yields results
    for pattern in mask_patterns:
        masks = list(data_dir.rglob(pattern))
        if masks:
            logger.info(f"✅ Found {len(masks)} masks with pattern '{pattern}'")
            break

    if not images or not masks:
        raise FileNotFoundError("Could not find images or masks in dataset")

    # Tokens indicating where to strip modality or descriptor suffixes
    image_tokens = [
        '_T1w', '_T1', '_t1', '_T2w', '_t2',
        '_flair', '_FLAIR', '_dwi', '_DWI',
        '_adc', '_ADC', '_image', '_brain'
    ]
    mask_tokens = ['_mask', '_lesion', '_label', '_seg', '_desc']

    def strip_at_first_token(name: str, tokens: list) -> str:
        """Return substring of `name` up to the earliest occurrence of any token."""
        indices = [name.find(tok) for tok in tokens if tok in name]
        if indices:
            idx = min(indices)
            return name[:idx]
        return name

    pairs = []
    lesion_counts = []

    # Attempt to pair each mask with its corresponding image
    for mask_path in masks:
        mask_base = strip_at_first_token(mask_path.stem, mask_tokens)
        match = None
        for img_path in images:
            img_base = strip_at_first_token(img_path.stem, image_tokens)
            # Fuzzy match: exact match or one prefix contained in the other
            if img_base == mask_base or img_base in mask_base or mask_base in img_base:
                match = img_path
                break
        if match:
            try:
                mask_obj = nib.load(str(mask_path))
                mask_data = mask_obj.get_fdata()
                has_lesion = np.any(mask_data > 0)
                pairs.append((match, mask_path))
                lesion_counts.append(1 if has_lesion else 0)
                del mask_obj, mask_data
                gc.collect()
            except Exception as e:
                logger.warning(f"Skipping {mask_path}: {e}")
                continue

    logger.info(f"📊 Created {len(pairs)} image–mask pairs")
    if lesion_counts:
        logger.info(f"🧠 Class balance: {np.mean(lesion_counts) * 100:.2f}% contain lesions")
    log_memory_usage("dataset_load_end")
    return pairs, np.array(lesion_counts)



def create_stratified_splits(pairs, lesion_presence, batch_size, test_size=0.1):
    """Generate train/validation splits compatible with batch size.

    We adapt the original function to ensure that each split contains a number
    of samples divisible by the batch size.  Stratification is performed
    based on lesion presence.
    """
    total_samples = len(pairs)
    test_samples = math.floor(total_samples * test_size)
    train_samples = total_samples - test_samples
    # Round down to nearest batch size
    test_samples = (test_samples // batch_size) * batch_size
    train_samples = total_samples - test_samples
    # Guarantee at least one batch in each split
    if test_samples < batch_size:
        test_samples = batch_size
        train_samples = total_samples - test_samples
    if train_samples < batch_size:
        train_samples = batch_size
        test_samples = total_samples - train_samples
    logger.info(
        f"🧮 Dataset split: Train={train_samples}"
        f" ({train_samples/total_samples*100:.1f}%), "
        f"Validation={test_samples} ({test_samples/total_samples*100:.1f}%)"
    )
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    for train_idx, test_idx in kf.split(pairs, lesion_presence):
        if len(test_idx) >= test_samples:
            test_idx = test_idx[:test_samples]
            break
    train_pairs = [pairs[i] for i in train_idx]
    test_pairs = [pairs[i] for i in test_idx]
    train_lesions = np.mean([lesion_presence[i] for i in train_idx])
    test_lesions = np.mean([lesion_presence[i] for i in test_idx])
    logger.info(
        f"⚖️ Lesion representation: Train={train_lesions*100:.1f}%, "
        f"Validation={test_lesions*100:.1f}%"
    )
    return train_pairs, test_pairs


# ---------------------------------------------------------------------------
# Data generator for variable volume sizes
# ---------------------------------------------------------------------------
class DynamicDataGenerator(tf.keras.utils.Sequence):
    """
    A memory‑efficient data generator that pads volumes to the maximum spatial
    shape defined in the configuration.  It supports on‑the‑fly augmentation
    and optional caching.  Images and masks must have identical spatial
    dimensions before padding.
    """
    def __init__(self, pairs, config: DynamicTrainingConfig, is_training=True):
        self.pair_paths = [(str(img), str(mask)) for img, mask in pairs]
        self.batch_size = config.BATCH_SIZE
        self.target_shape = config.INPUT_SHAPE[:-1]
        self.config = config
        self.is_training = is_training
        self.current_epoch = 0
        self.indexes = np.arange(len(self.pair_paths))
        self.executor = ThreadPoolExecutor(max_workers=min(4, self.batch_size))
        # Cache volumes if there is enough memory
        self._cache_enabled = psutil.virtual_memory().available > 50 * 1024**3
        self._volume_cache = {} if self._cache_enabled else None
        if self.is_training:
            np.random.shuffle(self.indexes)
        logger.info(
            f"🔧 Dynamic data generator: {len(self.pair_paths)} samples, "
            f"cache={'enabled' if self._cache_enabled else 'disabled'}"
        )

    def __len__(self):
        return len(self.pair_paths) // self.batch_size

    def __getitem__(self, index):
        batch_indexes = self.indexes[index * self.batch_size:(index + 1) * self.batch_size]
        batch_paths = [self.pair_paths[i] for i in batch_indexes]
        # Allocate batch arrays according to target_shape
        X = np.zeros((len(batch_paths), *self.target_shape, 1), dtype=np.float32)
        y = np.zeros((len(batch_paths), *self.target_shape, 1), dtype=np.float32)
        if self.is_training and len(batch_paths) > 1:
            futures = [
                self.executor.submit(self._load_sample_pair, img_path, mask_path, i)
                for i, (img_path, mask_path) in enumerate(batch_paths)
            ]
            for future in futures:
                i, img, mask = future.result()
                X[i, ..., 0] = img
                y[i, ..., 0] = mask
        else:
            for i, (img_path, mask_path) in enumerate(batch_paths):
                _, img, mask = self._load_sample_pair(img_path, mask_path, i)
                X[i, ..., 0] = img
                y[i, ..., 0] = mask
        gc.collect()
        return X, y

    def _load_sample_pair(self, img_path: str, mask_path: str, index: int):
        try:
            img = self._load_volume(img_path)
            mask = self._load_mask(mask_path)
            # Pad to target shape
            if img.shape != self.target_shape:
                img = self._pad_to_shape(img, self.target_shape)
            if mask.shape != self.target_shape:
                mask = self._pad_to_shape(mask, self.target_shape, is_mask=True)
            # Augment if training
            if self.is_training:
                img, mask = self.augment(img, mask)
            return index, img, mask
        except Exception as e:
            logger.warning(f"Error loading {img_path}: {e}")
            return index, np.zeros(self.target_shape, dtype=np.float32), np.zeros(self.target_shape, dtype=np.float32)

    def _pad_to_shape(self, volume, target_shape, is_mask=False):
        """Pad a smaller volume with zeros to reach the target shape"""
        output = np.zeros(target_shape, dtype=volume.dtype)
        copy_shape = tuple(min(v, t) for v, t in zip(volume.shape, target_shape))
        vol_slices = tuple(slice(0, s) for s in copy_shape)
        out_slices = tuple(slice(0, s) for s in copy_shape)
        output[out_slices] = volume[vol_slices]
        return output.astype(np.float32)

    @lru_cache(maxsize=32)
    def _load_volume(self, path: str) -> np.ndarray:
        if self._cache_enabled and path in self._volume_cache:
            return self._volume_cache[path].copy()
        try:
            img_obj = nib.load(path)
            img = img_obj.get_fdata().astype(np.float32)
            # No resizing; normalise only
            img = self.normalize(img)
            if self._cache_enabled and img.nbytes < 100 * 1024**2:
                self._volume_cache[path] = img.copy()
            return img
        except Exception as e:
            logger.warning(f"Error loading volume {path}: {e}")
            return np.zeros(self.target_shape, dtype=np.float32)

    def _load_mask(self, path: str) -> np.ndarray:
        try:
            mask_obj = nib.load(path)
            mask = mask_obj.get_fdata().astype(np.float32)
            mask = (mask > 0.5).astype(np.float32)
            return mask
        except Exception as e:
            logger.warning(f"Error loading mask {path}: {e}")
            return np.zeros(self.target_shape, dtype=np.float32)

    def normalize(self, img: np.ndarray) -> np.ndarray:
        if np.max(img) == 0:
            return np.zeros_like(img)
        non_zero = img[img > 0]
        if len(non_zero) == 0:
            return np.zeros_like(img)
        p1, p99 = np.percentile(non_zero, [1, 99])
        img = np.clip(img, p1, p99)
        mean_val = np.mean(non_zero)
        std_val = np.std(non_zero)
        if std_val > 0:
            img = (img - mean_val) / std_val
            img = (img - np.min(img)) / (np.max(img) - np.min(img) + 1e-8)
        else:
            img = (img - np.min(img)) / (np.max(img) - np.min(img) + 1e-8)
        return img.astype(np.float32)

    def enhance_lesions(self, mask: np.ndarray) -> np.ndarray:
        try:
            if np.sum(mask) == 0:
                return mask
            labeled = measure.label(mask > 0.5)
            regions = measure.regionprops(labeled)
            enhanced_mask = mask.copy()
            for region in regions:
                if region.area < self.config.SMALL_LESION_THRESHOLD:
                    lesion_mask = (labeled == region.label)
                    kernel = np.ones((3, 3, 3), dtype=bool)
                    expanded = binary_dilation(lesion_mask, structure=kernel, iterations=1)
                    enhanced_mask = np.logical_or(enhanced_mask, expanded).astype(np.float32)
            return enhanced_mask
        except Exception as e:
            logger.warning(f"Lesion enhancement failed: {e}")
            return mask

    def add_synthetic_lesion(self, img: np.ndarray, mask: np.ndarray):
        try:
            brain_mask = img > 0.1
            if np.sum(brain_mask) < 1000:
                return img, mask
            brain_coords = np.where(brain_mask)
            if len(brain_coords[0]) == 0:
                return img, mask
            idx = np.random.randint(0, len(brain_coords[0]))
            center = [brain_coords[i][idx] for i in range(3)]
            size = np.random.randint(2, min(8, self.config.SMALL_LESION_THRESHOLD // 10))
            intensity_factor = np.random.uniform(0.7, 1.3)
            lesion_shape = [
                size + np.random.randint(-1, 2),
                size + np.random.randint(-1, 2),
                max(1, size // 2 + np.random.randint(-1, 2))
            ]
            for i in range(3):
                start = max(0, center[i] - lesion_shape[i] // 2)
                end = min(img.shape[i], center[i] + lesion_shape[i] // 2)
                if start < end:
                    coords = np.mgrid[start:end, start:end, start:end]
                    distances = np.sqrt(
                        ((coords[0] - center[0]) / lesion_shape[0]) ** 2 +
                        ((coords[1] - center[1]) / lesion_shape[1]) ** 2 +
                        ((coords[2] - center[2]) / lesion_shape[2]) ** 2
                    )
                    lesion_region = distances <= 1.0
                    mask[start:end, start:end, start:end][lesion_region] = 1
                    original_intensity = img[start:end, start:end, start:end][lesion_region]
                    img[start:end, start:end, start:end][lesion_region] = (
                        original_intensity * intensity_factor
                    )
            img = np.clip(img, 0, 1)
        except Exception as e:
            logger.warning(f"Synthetic lesion creation failed: {e}")
        return img, mask

    def random_rotate(self, img: np.ndarray, mask: np.ndarray):
        try:
            angle = np.random.uniform(-self.config.ROTATION_RANGE, self.config.ROTATION_RANGE)
            axes_pairs = [(0, 1), (0, 2), (1, 2)]
            axes = random.choice(axes_pairs)
            img_rot = rotate(img, angle, axes=axes, reshape=False, order=1, mode='reflect', prefilter=False)
            mask_rot = rotate(mask, angle, axes=axes, reshape=False, order=0, mode='reflect', prefilter=False)
            return img_rot, mask_rot
        except Exception as e:
            logger.warning(f"Rotation failed: {e}")
            return img, mask

    def augment(self, img: np.ndarray, mask: np.ndarray):
        original_volume = np.sum(mask)
        # Spatial augmentations
        if np.random.rand() > 0.3:
            if np.random.rand() > 0.5:
                img, mask = self.random_rotate(img, mask)
            if np.random.rand() > 0.4:
                axis = np.random.choice([0, 1, 2])
                img = np.flip(img, axis=axis)
                mask = np.flip(mask, axis=axis)
        # Intensity augmentations
        if np.random.rand() > 0.2:
            if np.random.rand() > 0.3:
                brightness = np.random.uniform(0.8, 1.2)
                contrast = np.random.uniform(0.9, 1.1)
                img = img * contrast + (brightness - 1) * 0.5
                img = np.clip(img, 0, 1)
            if np.random.rand() > 0.6:
                noise_std = np.random.uniform(0.01, 0.05)
                noise = np.random.normal(0, noise_std, img.shape)
                img = np.clip(img + noise, 0, 1)
            if np.random.rand() > 0.7:
                gamma = np.random.uniform(0.7, 1.3)
                img = np.power(img, gamma)
        # Lesion enhancement
        if original_volume > 0:
            if original_volume < self.config.SMALL_LESION_THRESHOLD and np.random.rand() > 0.4:
                mask = self.enhance_lesions(mask)
        # Synthetic lesion creation
        if original_volume < self.config.SMALL_LESION_THRESHOLD // 2 and np.random.rand() < self.config.SYNTHETIC_LESION_PROB:
            img, mask = self.add_synthetic_lesion(img, mask)
        final_volume = np.sum(mask)
        if final_volume > original_volume * 3:
            logger.warning("Excessive lesion augmentation detected")
        return img.astype(np.float32), mask.astype(np.float32)

    def on_epoch_end(self):
        if self.is_training:
            np.random.shuffle(self.indexes)
        if self._cache_enabled and self.current_epoch % 10 == 0:
            self._volume_cache.clear()
            gc.collect()
            logger.info("🧹 Cache cleared for memory optimisation")
        self.current_epoch += 1
        if self.current_epoch % 5 == 0:
            log_memory_usage(f"epoch_{self.current_epoch}_end")

    def __del__(self):
        if hasattr(self, 'executor'):
            self.executor.shutdown(wait=False)


# ---------------------------------------------------------------------------
# Memory monitoring callback
# ---------------------------------------------------------------------------
class MemoryMonitoringCallback(tf.keras.callbacks.Callback):
    def __init__(self, log_frequency: int = 1):
        super().__init__()
        self.log_frequency = log_frequency
    def on_epoch_begin(self, epoch, logs=None):
        if epoch % self.log_frequency == 0:
            log_memory_usage(f"epoch_{epoch}_start")
    def on_batch_begin(self, batch, logs=None):
        if batch % 10 == 0:
            log_memory_usage(f"batch_{batch}")
    def on_epoch_end(self, epoch, logs=None):
        if epoch % self.log_frequency == 0:
            log_memory_usage(f"epoch_{epoch}_end")
            gc.collect()
            tf.keras.backend.clear_session()


# ---------------------------------------------------------------------------
# Build the segmentation model (UNet‑like skeleton with Mamba and SAM2)
# ---------------------------------------------------------------------------
def build_dynamic_model(config: DynamicTrainingConfig) -> tf.keras.Model:
    """Construct a UNet‑style segmentation model with dynamic input support."""
    inputs = tf.keras.Input(shape=config.INPUT_SHAPE)
    x = inputs
    # Encoder path
    skips = []
    filters = config.BASE_FILTERS
    for d in range(4):
        x = ResidualConvBlock(filters, kernel_reg=tf.keras.regularizers.l2(config.L2_REG))(x)
        x = VisionMambaBlock(filters)(x)
        skips.append(x)
        x = layers.MaxPool3D()(x)
        filters *= 2
    # Bottleneck with SAM2 attention and residual block
    x = ResidualConvBlock(filters, kernel_reg=tf.keras.regularizers.l2(config.L2_REG))(x)
    x = SAM2Attention(filters, heads=config.SAM_HEADS)(x)
    # Decoder path
    for d in reversed(range(4)):
        filters //= 2
        x = layers.UpSampling3D()(x)
        x = layers.Concatenate()([x, skips[d]])
        x = ResidualConvBlock(filters, kernel_reg=tf.keras.regularizers.l2(config.L2_REG))(x)
        x = VisionMambaBlock(filters)(x)
    # Output layer
    outputs = layers.Conv3D(1, 1, activation='sigmoid')(x)
    model = tf.keras.Model(inputs=inputs, outputs=outputs, name="SmartSOTA_Dynamic")
    return model


# ---------------------------------------------------------------------------
# Loss functions and metrics
# ---------------------------------------------------------------------------
def dice_coefficient(y_true, y_pred, smooth=1e-5):
    y_true_f = tf.reshape(y_true, [-1])
    y_pred_f = tf.reshape(y_pred, [-1])
    intersection = tf.reduce_sum(y_true_f * y_pred_f)
    return (2. * intersection + smooth) / (tf.reduce_sum(y_true_f) + tf.reduce_sum(y_pred_f) + smooth)

def dice_loss(y_true, y_pred):
    return 1 - dice_coefficient(y_true, y_pred)

def boundary_loss(y_true, y_pred):
    """
    Compute a boundary loss for 3‑D volumes by comparing finite differences
    (forward differences) along depth, height and width axes.
    """
    # Ensure inputs are float tensors
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)

    # Differences along the depth axis (axis=1)
    dy_true = y_true[:, 1:, :, :, :] - y_true[:, :-1, :, :, :]
    dy_pred = y_pred[:, 1:, :, :, :] - y_pred[:, :-1, :, :, :]

    # Differences along the height axis (axis=2)
    dx_true = y_true[:, :, 1:, :, :] - y_true[:, :, :-1, :, :]
    dx_pred = y_pred[:, :, 1:, :, :] - y_pred[:, :, :-1, :, :]

    # Differences along the width axis (axis=3)
    dz_true = y_true[:, :, :, 1:, :] - y_true[:, :, :, :-1, :]
    dz_pred = y_pred[:, :, :, 1:, :] - y_pred[:, :, :, :-1, :]

    # Compute squared difference of gradients and average over all voxels
    loss_d = tf.reduce_mean(tf.square(dy_true - dy_pred))
    loss_h = tf.reduce_mean(tf.square(dx_true - dx_pred))
    loss_w = tf.reduce_mean(tf.square(dz_true - dz_pred))

    return (loss_d + loss_h + loss_w) / 3.0


def combined_loss(y_true, y_pred, config: DynamicTrainingConfig):
    dl = dice_loss(y_true, y_pred)
    bl = boundary_loss(y_true, y_pred)
    return (config.DICE_LOSS_WEIGHT * dl + config.BOUNDARY_LOSS_WEIGHT * bl)


# ---------------------------------------------------------------------------
# Training pipeline
# ---------------------------------------------------------------------------
def train_dynamic_model(config: DynamicTrainingConfig):
    # Detect input shape
    max_dims = detect_input_shape(config.DATA_DIR)
    # Set config INPUT_SHAPE to max_dims with channel
    config.INPUT_SHAPE = max_dims + (1,)
    logger.info(f"🧭 INPUT_SHAPE set to: {config.INPUT_SHAPE}")
    # Load dataset and create splits
    pairs, lesion_presence = load_generic_dataset(config)
    train_pairs, val_pairs = create_stratified_splits(
        pairs, lesion_presence, config.BATCH_SIZE, test_size=config.VALIDATION_SPLIT
    )
    # Prepare data generators
    train_gen = DynamicDataGenerator(train_pairs, config, is_training=True)
    val_gen = DynamicDataGenerator(val_pairs, config, is_training=False)
    # Build model
    model = build_dynamic_model(config)
    model.summary(print_fn=logger.info)
    # Compile with Adam and custom loss
    optimizer = tf.keras.optimizers.Adam(learning_rate=config.INITIAL_LR)
    # LR schedule: linear warmup then cosine decay
    def lr_schedule(epoch):
        if epoch < config.WARMUP_EPOCHS:
            return config.INITIAL_LR * (epoch + 1) / config.WARMUP_EPOCHS
        else:
            progress = (epoch - config.WARMUP_EPOCHS) / max(1, config.TOTAL_EPOCHS - config.WARMUP_EPOCHS)
            cosine_decay = 0.5 * (1 + math.cos(math.pi * progress))
            return max(config.MIN_LR, config.INITIAL_LR * cosine_decay)
    lr_callback = tf.keras.callbacks.LearningRateScheduler(lr_schedule, verbose=0)
    memory_callback = MemoryMonitoringCallback(log_frequency=1)
    # Checkpoint callback
    checkpoint_cb = tf.keras.callbacks.ModelCheckpoint(
        filepath=str(config.checkpoint_path),
        monitor='val_loss',
        save_best_only=True,
        save_weights_only=False,
        mode='min',
        verbose=1
    )
    model.compile(
        optimizer=optimizer,
        loss=lambda y_true, y_pred: combined_loss(y_true, y_pred, config),
        metrics=[dice_coefficient]
    )
    logger.info("🚀 Starting training...")
    model.fit(
        train_gen,
        epochs=config.TOTAL_EPOCHS,
        validation_data=val_gen,
        callbacks=[lr_callback, memory_callback, checkpoint_cb],
        initial_epoch=config.INITIAL_EPOCH
    )
    # Save final model
    model.save(config.model_path)
    logger.info(f"🏁 Training complete. Model saved to {config.model_path}")


if __name__ == "__main__":
    # Instantiate configuration
    config = DynamicTrainingConfig()
    # Begin training
    train_dynamic_model(config)

2025-09-15 13:16:18,577 - SmartSOTA_Dynamic - INFO - ✅ All imports successful
2025-09-15 13:16:18,577 - SmartSOTA_Dynamic - INFO - TensorFlow eager execution: True
2025-09-15 13:16:18,578 - SmartSOTA_Dynamic - INFO - Environment verified:
- Python 3.10.14 | packaged by conda-forge | (main, Mar 20 2024, 12:45:18) [GCC 12.3.0]
- TensorFlow 2.15.0
- NumPy 1.25.2
- GPU devices: 0
2025-09-15 13:16:18,582 - SmartSOTA_Dynamic - INFO - 📋 DynamicTrainingConfig initialised:
   Data directory: /home/rbielski/Atlas_2/Training_Split/Training_Set
   Batch size: 2
   Warmup epochs: 15
   Minimum LR: 5e-07
   Dice/boundary weights: 0.4:0.6

2025-09-15 13:16:18,583 - SmartSOTA_Dynamic - INFO - 🔍 Detecting input shape from dataset…
pixdim[0] (qfac) should be 1 (default) or -1; setting qfac to 1
2025-09-15 13:16:18,595 - nibabel.global - INFO - pixdim[0] (qfac) should be 1 (default) or -1; setting qfac to 1
pixdim[0] (qfac) should be 1 (default) or -1; setting qfac to 1
2025-09-15 13:16:18,616 - nibabel.

Epoch 1/200


2025-09-15 13:18:23,008 - SmartSOTA_Dynamic - INFO - Memory at batch_0: CPU=1.12GB | GPU mem tracking failed | Disk: 1760.3GB free
pixdim[0] (qfac) should be 1 (default) or -1; setting qfac to 1
2025-09-15 13:18:30,569 - nibabel.global - INFO - pixdim[0] (qfac) should be 1 (default) or -1; setting qfac to 1
pixdim[0] (qfac) should be 1 (default) or -1; setting qfac to 1
2025-09-15 13:18:30,614 - nibabel.global - INFO - pixdim[0] (qfac) should be 1 (default) or -1; setting qfac to 1
pixdim[0] (qfac) should be 1 (default) or -1; setting qfac to 1
2025-09-15 13:18:33,178 - nibabel.global - INFO - pixdim[0] (qfac) should be 1 (default) or -1; setting qfac to 1
pixdim[0] (qfac) should be 1 (default) or -1; setting qfac to 1
2025-09-15 13:18:33,261 - nibabel.global - INFO - pixdim[0] (qfac) should be 1 (default) or -1; setting qfac to 1
pixdim[0] (qfac) should be 1 (default) or -1; setting qfac to 1
2025-09-15 13:18:36,602 - nibabel.global - INFO - pixdim[0] (qfac) should be 1 (default) or -

  1/209 [..............................] - ETA: 7:05:02 - loss: 1.4955 - dice_coefficient: 0.0066

pixdim[0] (qfac) should be 1 (default) or -1; setting qfac to 1
2025-09-15 13:20:26,171 - nibabel.global - INFO - pixdim[0] (qfac) should be 1 (default) or -1; setting qfac to 1
pixdim[0] (qfac) should be 1 (default) or -1; setting qfac to 1
2025-09-15 13:20:26,274 - nibabel.global - INFO - pixdim[0] (qfac) should be 1 (default) or -1; setting qfac to 1


...

pixdim[0] (qfac) should be 1 (default) or -1; setting qfac to 1
2025-09-15 13:24:02,311 - nibabel.global - INFO - pixdim[0] (qfac) should be 1 (default) or -1; setting qfac to 1
pixdim[0] (qfac) should be 1 (default) or -1; setting qfac to 1
2025-09-15 13:24:02,489 - nibabel.global - INFO - pixdim[0] (qfac) should be 1 (default) or -1; setting qfac to 1


  4/209 [..............................] - ETA: 6:08:57 - loss: 1.4960 - dice_coefficient: 0.0018

pixdim[0] (qfac) should be 1 (default) or -1; setting qfac to 1
2025-09-15 13:25:50,168 - nibabel.global - INFO - pixdim[0] (qfac) should be 1 (default) or -1; setting qfac to 1
pixdim[0] (qfac) should be 1 (default) or -1; setting qfac to 1
2025-09-15 13:25:50,256 - nibabel.global - INFO - pixdim[0] (qfac) should be 1 (default) or -1; setting qfac to 1


  5/209 [..............................] - ETA: 6:06:48 - loss: 1.4960 - dice_coefficient: 0.0016

pixdim[0] (qfac) should be 1 (default) or -1; setting qfac to 1
2025-09-15 13:27:37,711 - nibabel.global - INFO - pixdim[0] (qfac) should be 1 (default) or -1; setting qfac to 1
pixdim[0] (qfac) should be 1 (default) or -1; setting qfac to 1
2025-09-15 13:27:37,820 - nibabel.global - INFO - pixdim[0] (qfac) should be 1 (default) or -1; setting qfac to 1
